# Task
Confirm the successful upload of "Submittal and RFI Tracker Lists (1).xlsx", the listing of its sheet names, the reading of the "Prolog Submittals " worksheet with the 5th row as the header, and the application of forward-fill to handle empty cells, then offer to assist with further analysis or tasks based on the processed data.

## Upload Excel File

### Subtask:
Upload a single Excel file from your local drive.


**Reasoning**:
The user wants to upload an Excel file from their local drive using `google.colab.files.upload()`. This command will open a file selection dialog in the Colab environment.



In [20]:
from google.colab import files

print("Please upload the 'Submittal and RFI Tracker Lists (1).xlsx' file.")
uploaded = files.upload()
uploaded_file_name = list(uploaded.keys())[0]
print(f"File '{uploaded_file_name}' uploaded successfully.")

Please upload the 'Submittal and RFI Tracker Lists (1).xlsx' file.


Saving Submittal and RFI Tracker Lists.xlsx to Submittal and RFI Tracker Lists (3).xlsx
File 'Submittal and RFI Tracker Lists (3).xlsx' uploaded successfully.


**Reasoning**:
The Excel file has been successfully uploaded. The next step is to list the sheet names within the uploaded Excel file to understand its structure.



In [21]:
import pandas as pd

xls = pd.ExcelFile(uploaded_file_name)
sheet_names = xls.sheet_names
print("Sheet names in the Excel file:")
for i, name in enumerate(sheet_names):
    print(f"{i+1}. {name}")

Sheet names in the Excel file:
1. Summary Chart
2. Delay Analysis
3. Delay in Response
4. Prolog Submittals 
5. RFIs
6. Doc Typ
7. Discipline
8. Project Prefix querries
9. Public Holiday
10. Project Prefix
11. Status Legend
12. Testing
13. draft


/usr/local/lib/python3.12/dist-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Delay in Response'!$A:$P.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


## Select and Load Another Worksheet

Now you can interactively select any other worksheet from the uploaded Excel file. The selected sheet will be loaded into a new DataFrame (`df_selected_sheet`), with the 5th row as the header and forward-fill applied to handle empty cells.

In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create a dropdown widget for sheet selection
sheet_selector = widgets.Dropdown(
    options=sheet_names,
    description='Select a different sheet:',
    disabled=False,
    value=None # Set initial value to None so the user explicitly selects
)

def on_sheet_select(change):
    clear_output(wait=True)
    selected_sheet = change.new
    if selected_sheet:
        try:
            global df_selected_sheet_filled # Declare df_selected_sheet_filled as global
            # Load the selected sheet into a DataFrame, applying header=4 and ffill
            df_selected_sheet = pd.read_excel(uploaded_file_name, sheet_name=selected_sheet, header=4)
            df_selected_sheet_filled = df_selected_sheet.ffill()
            print(f"\nSuccessfully loaded and forward-filled data from worksheet: '{selected_sheet}'")
            print("First 5 rows of the processed DataFrame for the selected sheet:")
            display(df_selected_sheet_filled.head())
        except Exception as e:
            print(f"Error loading sheet '{selected_sheet}': {e}")
    else:
        print("Please select a sheet.")

sheet_selector.observe(on_sheet_select, names='value')

print("Please select a sheet from the dropdown below:")
display(sheet_selector)

# Note: df_selected_sheet_filled will be updated each time a new selection is made
# You can then use df_selected_sheet_filled for further analysis on the newly selected sheet.

/usr/local/lib/python3.12/dist-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Delay in Response'!$A:$P.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)



Successfully loaded and forward-filled data from worksheet: 'Prolog Submittals '
First 5 rows of the processed DataFrame for the selected sheet:


,Doc ID,1st Submission\nDate,Latest Submission Date,Latest Revision,Latest Approval Status,Approval Code,# of Submissions,Prolog Submittal No.,Submitted by,Document Title,...,Remark,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,FOR FOLLOW UP,Unnamed: 48,Unnamed: 49,Unnamed: 50
0,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,Rev.1,Approved with Comments,AWC,2.0,1.0,Soe Myat,Project Quality Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,Soe Myat,NaN,1.0,1.0
1,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,Rev.1,Approved with Comments,AWC,2.0,1.0,Radha,Project Quality Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,Radha,NaN,1.0,0.0
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15,Rev.0,Approved with Comments,AWC,1.0,2.0,Shena,Project Risk Management Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,Shena,NaN,1.0,1.0
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-10-30,Rev.2,Approved,APP,2.0,2.0,Max/Edwin,Project Risk Management Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,Max/Edwin,NaN,1.0,1.0
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05,Rev.,Approved with Comments,AWC,1.0,2.0,Max/Edwin,Project Risk Management Plan,...,NaN,NaN,NaN,NaN,NaN,NaN,Max/Edwin,NaN,1.0,1.0


## Consolidate Data Loading and Cleaning

### Subtask:
Consolidate the steps for loading data with a column limit, removing empty rows, and removing empty columns into a single code cell. The resulting DataFrame will be named `df_cleaned_and_filtered`.

In [28]:
import string

# Generate a list of Excel column names up to 'AP'
alphabet = list(string.ascii_uppercase)
excel_cols = []
for i in range(2):
    for char in alphabet:
        if i == 0:
            excel_cols.append(char)
        else:
            excel_cols.append(alphabet[i-1] + char)
        if (i*26 + alphabet.index(char)) >= 41: # 'AP' is the 42nd column, index 41
            break

# 1. Load the selected sheet into a DataFrame, applying header=4, ffill, and limiting columns by integer index
df_cleaned_and_filtered = pd.read_excel(
    uploaded_file_name,
    sheet_name=selected_sheet,
    header=4,
    usecols=range(len(excel_cols)) # Use integer indices to select columns by position
)
df_cleaned_and_filtered = df_cleaned_and_filtered.ffill()

# 2. Remove empty rows
df_cleaned_and_filtered = df_cleaned_and_filtered.dropna(how='all')

# 3. Remove empty columns
df_cleaned_and_filtered = df_cleaned_and_filtered.dropna(axis=1, how='all')

print(f"\nSuccessfully loaded, forward-filled, and cleaned data from worksheet: '{selected_sheet}' up to column 'AP'.")
print("First 5 rows of the consolidated and cleaned DataFrame:")
display(df_cleaned_and_filtered.head())

/usr/local/lib/python3.12/dist-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Delay in Response'!$A:$P.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)



Successfully loaded, forward-filled, and cleaned data from worksheet: 'Prolog Submittals ' up to column 'AP'.
First 5 rows of the consolidated and cleaned DataFrame:


,Doc ID,1st Submission\nDate,Latest Submission Date,Latest Revision,Latest Approval Status,Approval Code,# of Submissions,Prolog Submittal No.,Submitted by,Document Title,...,Actual Date S.O. Response,SO Response Variance,SO Review Status,To Resubmit (Yes/No),Date CES to Response\n(14 Working Days),Closed,Overdue to resubmit,Delays to resubmit,Target date to resubmit,Remark
0,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,Rev.1,Approved with Comments,AWC,2.0,1.0,Soe Myat,Project Quality Plan,...,2023-12-05,122,Approved with Comments,YES,2023-12-26,YES,Resubmitted,NaN,NaN,NaN
1,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,Rev.1,Approved with Comments,AWC,2.0,1.0,Radha,Project Quality Plan,...,2024-09-10,69,Approved with Comments,YES,2024-09-30,NO,Overdue,343.0,NaN,NaN
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15,Rev.0,Approved with Comments,AWC,1.0,2.0,Shena,Project Risk Management Plan,...,2023-06-14,1,Approved with Comments,YES,2023-07-05,YES,Resubmitted,343.0,NaN,NaN
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-10-30,Rev.2,Approved,APP,2.0,2.0,Max/Edwin,Project Risk Management Plan,...,2023-09-13,1,Approved with Comments,YES,2023-10-03,YES,Resubmitted,343.0,NaN,NaN
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05,Rev.,Approved with Comments,AWC,1.0,2.0,Max/Edwin,Project Risk Management Plan,...,2023-09-13,1,Approved with Comments,YES,2023-10-03,YES,Resubmitted,343.0,NaN,NaN


# Task
Analyze the 'Doc ID' column in `df_cleaned_and_filtered` for patterns, unique values, and common separators to understand its structure and how it might be derived from other columns, then display the findings.

## Analyze 'Doc ID' Column for Patterns

### Subtask:
Inspect the `Doc ID` column in `df_cleaned_and_filtered` for any discernible patterns, unique values, common separators, or potential components that might indicate its derivation from other columns.


**Reasoning**:
To understand the patterns, unique values, and common separators within the 'Doc ID' column, I will perform the requested inspections using Python code, including displaying unique values, counts, frequencies, and a sample of entries.



In [29]:
print("1. First 10 unique values from 'Doc ID' column:")
print(df_cleaned_and_filtered['Doc ID'].unique()[:10])

print("\n2. Total number of unique 'Doc ID' values and total rows:")
unique_doc_ids = df_cleaned_and_filtered['Doc ID'].nunique()
total_rows = len(df_cleaned_and_filtered)
print(f"Total unique 'Doc ID' values: {unique_doc_ids}")
print(f"Total rows in DataFrame: {total_rows}")

print("\n3. Frequency of the top 10 most common 'Doc ID' values:")
print(df_cleaned_and_filtered['Doc ID'].value_counts().head(10))

print("\n4. Sample of 10 random 'Doc ID' entries for separator inspection:")
print(df_cleaned_and_filtered['Doc ID'].sample(10).tolist())

1. First 10 unique values from 'Doc ID' column:
['131242-WST00-PP-PM-0001' '131242-WST00-PP-PC-0001'
 '131242-WSW41-PP-PC-0001' '#000002.0_ Reply_2023 08 31----'
 '#000002.0_ Reply_2023 09 13 (002)_CES reply_20231025----'
 '131242-WST00-PP-IM-0001' '131242-WSW41-PP-IM-0001'
 '131242-WST00-PP-HS-0001' '131242-WST00-PG-PC-0001'
 '131242-WSW41-PG-PC-0001']

2. Total number of unique 'Doc ID' values and total rows:
Total unique 'Doc ID' values: 5952
Total rows in DataFrame: 11099

3. Frequency of the top 10 most common 'Doc ID' values:
Doc ID
131242-WST02-DR-A-2007_FSSD_BP    9
131242-WST02-DR-A-2001_FSSD_BP    9
131242-WST02-DR-A-1001_PUB        9
131242-WST02-DR-A-1001_FSSD_BP    9
131242-WST02-DR-A-1000_FSSD_BP    9
131242-WST02-DR-A-1002_FSSD_BP    9
131242-WST02-DR-A-1004_FSSD_BP    9
131242-WST02-DR-A-1003_FSSD_BP    9
131242-WST02-DR-A-2002_FSSD_BP    9
131242----                        9
Name: count, dtype: int64

4. Sample of 10 random 'Doc ID' entries for separator inspection:
['

## Define 'Doc ID' Calculation Formula

### Subtask:
Since the 'Doc ID' values may be derived from a specific formula or combination of other columns in the original Excel file, and `pandas` only reads the resulting values, it's necessary to obtain this formula directly. This information is crucial for accurately re-calculating or validating the 'Doc ID' column.


### Subtask: Define 'Doc ID' Calculation Formula

**Action Required:**

To accurately understand and potentially reproduce the 'Doc ID' values, we need the formula or logic used to generate them in the original Excel file. Please provide this information, considering the patterns observed in the previous step (e.g., common prefixes like '131242', segments separated by hyphens, or descriptive text).

**Please specify:**

1.  **The exact formula or logic:** For example, it might be a concatenation of several columns (e.g., `='Prefix'-'Type'-'Number'`) or involve conditional logic. Be as precise as possible.
2.  **Which columns are used:** List the names of the original Excel columns that contribute to the 'Doc ID'.
3.  **How they are combined or transformed:** Describe the operations performed (e.g., concatenation with hyphens, specific text extractions, or lookup references).

If the exact formula is unknown, please provide any details you have about how the 'Doc ID' is constructed (e.g., 'It looks like the first part is always '131242', followed by values from columns 'A', 'B', and 'C' separated by hyphens'). This information is critical for further analysis.

## Re-calculate and Update 'Doc ID'

### Subtask:
Based on the formula or logic provided by the user, apply the same calculation to create or update the 'Doc ID' column in the `df_cleaned_and_filtered` DataFrame. Display the head of the DataFrame to show the updated column.


In [31]:
df_cleaned_and_filtered['Recalculated Doc ID'] = (
    df_cleaned_and_filtered['Proj. Code'].astype(str) + '-' +
    df_cleaned_and_filtered['Proj. Prefix'].astype(str) + '-' +
    df_cleaned_and_filtered['Doc Type'].astype(str) + '-' +
    df_cleaned_and_filtered['Discipline'].astype(str) + '-' +
    df_cleaned_and_filtered['Number'].astype(str)
)

print("DataFrame with 'Recalculated Doc ID' column:")
display(df_cleaned_and_filtered.head())

DataFrame with 'Recalculated Doc ID' column:


,Doc ID,1st Submission\nDate,Latest Submission Date,Latest Revision,Latest Approval Status,Approval Code,# of Submissions,Prolog Submittal No.,Submitted by,Document Title,...,SO Response Variance,SO Review Status,To Resubmit (Yes/No),Date CES to Response\n(14 Working Days),Closed,Overdue to resubmit,Delays to resubmit,Target date to resubmit,Remark,Recalculated Doc ID
0,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,Rev.1,Approved with Comments,AWC,2.0,1.0,Soe Myat,Project Quality Plan,...,122,Approved with Comments,YES,2023-12-26,YES,Resubmitted,NaN,NaN,NaN,131242-WST00-PP-PM-0001
1,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,Rev.1,Approved with Comments,AWC,2.0,1.0,Radha,Project Quality Plan,...,69,Approved with Comments,YES,2024-09-30,NO,Overdue,343.0,NaN,NaN,131242-WST00-PP-PM-0001
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15,Rev.0,Approved with Comments,AWC,1.0,2.0,Shena,Project Risk Management Plan,...,1,Approved with Comments,YES,2023-07-05,YES,Resubmitted,343.0,NaN,NaN,131242-WST00-PP-PC-0001
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-10-30,Rev.2,Approved,APP,2.0,2.0,Max/Edwin,Project Risk Management Plan,...,1,Approved with Comments,YES,2023-10-03,YES,Resubmitted,343.0,NaN,NaN,131242-WSW41-PP-PC-0001
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05,Rev.,Approved with Comments,AWC,1.0,2.0,Max/Edwin,Project Risk Management Plan,...,1,Approved with Comments,YES,2023-10-03,YES,Resubmitted,343.0,NaN,NaN,#000002.0_ Reply_2023 08 31-WSW41-PP-PC-0001


**Reasoning**:
To validate the accuracy of the inferred formula, I need to compare the newly created 'Recalculated Doc ID' with the original 'Doc ID' column. This comparison will help determine if the inferred formula correctly reproduces the original 'Doc ID' values.



In [32]:
mismatched_doc_ids = df_cleaned_and_filtered[df_cleaned_and_filtered['Doc ID'] != df_cleaned_and_filtered['Recalculated Doc ID']]

total_rows = len(df_cleaned_and_filtered)
matches = total_rows - len(mismatched_doc_ids)
match_percentage = (matches / total_rows) * 100
mismatch_percentage = (len(mismatched_doc_ids) / total_rows) * 100

print(f"Total rows: {total_rows}")
print(f"Matching 'Doc ID' and 'Recalculated Doc ID': {matches} ({match_percentage:.2f}%)")
print(f"Mismatched 'Doc ID' and 'Recalculated Doc ID': {len(mismatched_doc_ids)} ({mismatch_percentage:.2f}%)\n")

if not mismatched_doc_ids.empty:
    print("Sample of rows where 'Doc ID' and 'Recalculated Doc ID' do not match:")
    display(mismatched_doc_ids[['Doc ID', 'Recalculated Doc ID', 'Proj. Code', 'Proj. Prefix', 'Doc Type', 'Discipline', 'Number']].head())
else:
    print("All 'Doc ID' and 'Recalculated Doc ID' values match.")

Total rows: 11099
Matching 'Doc ID' and 'Recalculated Doc ID': 10992 (99.04%)
Mismatched 'Doc ID' and 'Recalculated Doc ID': 107 (0.96%)

Sample of rows where 'Doc ID' and 'Recalculated Doc ID' do not match:


,Doc ID,Recalculated Doc ID,Proj. Code,Proj. Prefix,Doc Type,Discipline,Number
4,#000002.0_ Reply_2023 08 31----,#000002.0_ Reply_2023 08 31-WSW41-PP-PC-0001,#000002.0_ Reply_2023 08 31,WSW41,PP,PC,0001
6,#000002.0_ Reply_2023 09 13 (002)_CES reply_20...,#000002.0_ Reply_2023 09 13 (002)_CES reply_20...,#000002.0_ Reply_2023 09 13 (002)_CES reply_20...,WSW41,PP,PC,0001
26,Reply to Comment Sheet_#000007----,Reply to Comment Sheet_#000007-WST00-PG-PC-0002,Reply to Comment Sheet_#000007,WST00,PG,PC,0002
61,131242--LT-PM-Reply to Comment Sheet_#000025,131242-WST00-LT-PM-Reply to Comment Sheet_#000025,131242,WST00,LT,PM,Reply to Comment Sheet_#000025
96,Comment Reply Sheet 08112023----,Comment Reply Sheet 08112023-WSW41-RT-PM-0002,Comment Reply Sheet 08112023,WSW41,RT,PM,0002


### Analysis of 'Doc ID' Recalculation Results

Our attempt to re-calculate the 'Doc ID' based on the inferred formula (`'Proj. Code' + '-' + 'Proj. Prefix' + '-' + 'Doc Type' + '-' + 'Discipline' + '-' + 'Number'`) has shown significant success, with **99.04% of the 'Doc ID' values matching** the original ones. This indicates that the assumed formula is largely correct for the majority of the data.

However, **0.96% (107 rows) of the 'Doc ID' values still do not match**. The sample of mismatched rows reveals some interesting patterns:

*   **Descriptive Text in 'Doc ID'**: Some 'Doc ID' values are not strictly structured codes but include descriptive text (e.g., `#000002.0_ Reply_2023 08 31----` or `Reply to Comment Sheet_#000007----`). In these cases, it appears that the 'Proj. Code' column itself might contain this descriptive text, leading to a 'Recalculated Doc ID' that is a mix of the descriptive text and the concatenated code components.
*   **Variations in Delimiters or Components**: For some entries, the original 'Doc ID' might use different separators or include/exclude certain components compared to our inferred formula.
*   **'Number' Column Content**: In some cases, the 'Number' column seems to contain descriptive text as well (e.g., `'Reply to Comment Sheet_#000025'`), which further complicates a simple concatenation.

To address these mismatches, we can consider the following approaches:

1.  **Conditional Logic**: Implement conditional logic to apply different 'Doc ID' formulas based on specific patterns found in the 'Doc ID' or related columns. For instance, if a 'Doc ID' starts with '#', it might follow a different construction rule.
2.  **Text Extraction/Cleaning**: For descriptive 'Doc ID's, extract only the coding portion if a structured code is desired, or acknowledge that some 'Doc ID's are inherently descriptive.
3.  **User Input Refinement**: Revisit the 'Doc ID' generation logic with the user, presenting these edge cases for clarification on how these specific 'Doc ID's are formed in the original system.

Would you like to refine the formula to account for these specific mismatches, or should we proceed with the current understanding, acknowledging that a small percentage will remain different?

## Convert 'Date Submit' Column to Datetime

### Subtask:
Convert the 'Date Submit' column in `df_cleaned_and_filtered` to datetime objects, coercing any unparseable dates to `NaT`.

In [33]:
df_cleaned_and_filtered['Date Submit'] = pd.to_datetime(df_cleaned_and_filtered['Date Submit'], errors='coerce')
print("Data type of 'Date Submit' column after conversion:")
df_cleaned_and_filtered.info()

Data type of 'Date Submit' column after conversion:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11099 entries, 0 to 11098
Data columns (total 42 columns):
 #   Column                                                     Non-Null Count  Dtype         
---  ------                                                     --------------  -----         
 0   Doc ID                                                     11099 non-null  object        
 1   1st Submission
Date                                        11099 non-null  datetime64[ns]
 2   Latest Submission Date                                     11099 non-null  datetime64[ns]
 3   Latest Revision                                            11099 non-null  object        
 4   Latest Approval Status                                     11099 non-null  object        
 5   Approval Code                                              11099 non-null  object        
 6   # of Submissions                                           11099 non-null 

## Update '1st Submission Date'

### Subtask:
Update the '1st Submission Date' column in `df_cleaned_and_filtered` by finding the earliest date in 'Date Submit' for each 'Doc ID'.

In [34]:
# Calculate the earliest 'Date Submit' for each 'Doc ID'
earliest_submission_dates = df_cleaned_and_filtered.groupby('Doc ID')['Date Submit'].min().reset_index()
earliest_submission_dates.rename(columns={'Date Submit': 'Earliest Date Submit'}, inplace=True)

# Merge this back into the original DataFrame
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    earliest_submission_dates,
    on='Doc ID',
    how='left'
)

# Update the '1st Submission Date' column
df_cleaned_and_filtered['1st Submission\nDate'] = df_cleaned_and_filtered['Earliest Date Submit']

# Drop the temporary 'Earliest Date Submit' column
df_cleaned_and_filtered.drop(columns=['Earliest Date Submit'], inplace=True)

print("DataFrame after updating '1st Submission Date' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Date Submit', '1st Submission\nDate']].head())

DataFrame after updating '1st Submission Date' column:


,Doc ID,Date Submit,1st Submission\nDate
0,131242-WST00-PP-PM-0001,2023-05-15,2023-05-15
1,131242-WST00-PP-PM-0001,2024-05-13,2023-05-15
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-09-05
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05


## Update 'Latest Submission Date'

### Subtask:
Update the 'Latest Submission Date' column in `df_cleaned_and_filtered` by finding the latest date in 'Date Submit' for each 'Doc ID'.

In [35]:
# Calculate the latest 'Date Submit' for each 'Doc ID'
latest_submission_dates = df_cleaned_and_filtered.groupby('Doc ID')['Date Submit'].max().reset_index()
latest_submission_dates.rename(columns={'Date Submit': 'Latest Date Submit'}, inplace=True)

# Merge this back into the original DataFrame
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    latest_submission_dates,
    on='Doc ID',
    how='left'
)

# Update the 'Latest Submission Date' column
df_cleaned_and_filtered['Latest Submission Date'] = df_cleaned_and_filtered['Latest Date Submit']

# Drop the temporary 'Latest Date Submit' column
df_cleaned_and_filtered.drop(columns=['Latest Date Submit'], inplace=True)

print("DataFrame after updating 'Latest Submission Date' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Date Submit', 'Latest Submission Date']].head())

DataFrame after updating 'Latest Submission Date' column:


,Doc ID,Date Submit,Latest Submission Date
0,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13
1,131242-WST00-PP-PM-0001,2024-05-13,2024-05-13
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-10-30
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05


## Update 'Latest Revision'

### Subtask:
Update the 'Latest Revision' column in `df_cleaned_and_filtered` by getting the value from the 'Rev ' column that corresponds to the 'Latest Submission Date' for each 'Doc ID', and then affix 'Rev.' to the found value.

## Update 'Latest Approval Status'

### Subtask:
Update the 'Latest Approval Status' column in `df_cleaned_and_filtered` by getting the value from the 'SO Review Status' column that corresponds to the 'Latest Submission Date' for each 'Doc ID'.

In [37]:
# Group by 'Doc ID' and find the index of the row with the maximum 'Date Submit'
idx = df_cleaned_and_filtered.groupby('Doc ID')['Date Submit'].idxmax()

# Get the 'Doc ID' and corresponding 'SO Review Status' for the latest submission date
latest_approval_status_per_doc_id = df_cleaned_and_filtered.loc[idx, ['Doc ID', 'SO Review Status']]

# Rename 'SO Review Status' column for clarity during merge
latest_approval_status_per_doc_id.rename(columns={'SO Review Status': 'Latest Approval Status Value'}, inplace=True)

# Merge this back into the main DataFrame based on 'Doc ID'
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    latest_approval_status_per_doc_id,
    on='Doc ID',
    how='left'
)

# Update the 'Latest Approval Status' column
df_cleaned_and_filtered['Latest Approval Status'] = df_cleaned_and_filtered['Latest Approval Status Value']

# Drop the temporary column
df_cleaned_and_filtered.drop(columns=['Latest Approval Status Value'], inplace=True)

print("DataFrame after updating 'Latest Approval Status' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Date Submit', 'Latest Submission Date', 'SO Review Status', 'Latest Approval Status']].head())

DataFrame after updating 'Latest Approval Status' column:


,Doc ID,Date Submit,Latest Submission Date,SO Review Status,Latest Approval Status
0,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,Approved with Comments,Approved with Comments
1,131242-WST00-PP-PM-0001,2024-05-13,2024-05-13,Approved with Comments,Approved with Comments
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15,Approved with Comments,Approved with Comments
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-10-30,Approved with Comments,Approved
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05,Approved with Comments,Approved with Comments


In [36]:
# Group by 'Doc ID' and find the index of the row with the maximum 'Date Submit'
idx = df_cleaned_and_filtered.groupby('Doc ID')['Date Submit'].idxmax()

# Get the 'Doc ID' and corresponding 'Rev ' for the latest submission date
latest_rev_per_doc_id = df_cleaned_and_filtered.loc[idx, ['Doc ID', 'Rev ']]

# Rename 'Rev ' column for clarity during merge
latest_rev_per_doc_id.rename(columns={'Rev ': 'Latest Rev Value'}, inplace=True)

# Merge this back into the main DataFrame based on 'Doc ID'
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    latest_rev_per_doc_id,
    on='Doc ID',
    how='left'
)

# Affix "Rev." and update the 'Latest Revision' column
df_cleaned_and_filtered['Latest Revision'] = 'Rev.' + df_cleaned_and_filtered['Latest Rev Value'].astype(str)

# Drop the temporary column
df_cleaned_and_filtered.drop(columns=['Latest Rev Value'], inplace=True)

print("DataFrame after updating 'Latest Revision' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Date Submit', 'Latest Submission Date', 'Rev ', 'Latest Revision']].head())

DataFrame after updating 'Latest Revision' column:


,Doc ID,Date Submit,Latest Submission Date,Rev,Latest Revision
0,131242-WST00-PP-PM-0001,2023-05-15,2024-05-13,0,Rev.1
1,131242-WST00-PP-PM-0001,2024-05-13,2024-05-13,1,Rev.1
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15,0,Rev.0
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-10-30,1,Rev.2
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05,1,Rev.1


## Update 'Approval Code'

### Subtask:
Update the 'Approval Code' column in `df_cleaned_and_filtered` based on the 'Latest Approval Status' using the following rules:
- "REJ" for "Rejected" or "Not Approved - Revise and resubmit"
- "AWC" for "Approved with Comments" or "Approved as noted"
- "APP" for "For Information" or "Approved"
- "Pending" for "Awaiting S.O.'s response", or if 'Latest Approval Status' is empty or 0
- "VOID" for "(VOID / NOT IN USE)"
- "To Check" for any other cases.

In [38]:
def get_approval_code(status):
    status_str = str(status).strip()
    if status_str in ["Rejected", "Not Approved - Revise and resubmit"]:
        return "REJ"
    elif status_str in ["Approved with Comments", "Approved as noted"]:
        return "AWC"
    elif status_str in ["For Information", "Approved"]:
        return "APP"
    elif status_str in ["Awaiting S.O.'s response", "", "0"] or pd.isna(status):
        return "Pending"
    elif status_str == "(VOID / NOT IN USE)":
        return "VOID"
    else:
        return "To Check"

df_cleaned_and_filtered['Approval Code'] = df_cleaned_and_filtered['Latest Approval Status'].apply(get_approval_code)

print("DataFrame after updating 'Approval Code' column:")
display(df_cleaned_and_filtered[['Latest Approval Status', 'Approval Code']].head())

DataFrame after updating 'Approval Code' column:


,Latest Approval Status,Approval Code
0,Approved with Comments,AWC
1,Approved with Comments,AWC
2,Approved with Comments,AWC
3,Approved,APP
4,Approved with Comments,AWC


## Update '# of Submissions'

### Subtask:
Update the '# of Submissions' column in `df_cleaned_and_filtered` based on the counts of each 'Doc ID' in the DataFrame.

In [39]:
# Calculate the number of submissions for each 'Doc ID'
submission_counts = df_cleaned_and_filtered.groupby('Doc ID')['Doc ID'].transform('count')

# Update the '# of Submissions' column
df_cleaned_and_filtered['# of Submissions'] = submission_counts

print("DataFrame after updating '# of Submissions' column:")
display(df_cleaned_and_filtered[['Doc ID', '# of Submissions']].head())

DataFrame after updating '# of Submissions' column:


,Doc ID,# of Submissions
0,131242-WST00-PP-PM-0001,2
1,131242-WST00-PP-PM-0001,2
2,131242-WST00-PP-PC-0001,1
3,131242-WSW41-PP-PC-0001,2
4,#000002.0_ Reply_2023 08 31----,1


## Update 'Prolog Submittal No.'

### Subtask:
Update the 'Prolog Submittal No.' column in `df_cleaned_and_filtered` by getting unique values from 'Prolog Submittal No..1' for each 'Doc ID'. If multiple different values are found, separate them with a comma.

In [41]:
# Group by 'Doc ID' and aggregate unique 'Prolog Submittal No..1' values
consolidated_prolog_submittal_no = df_cleaned_and_filtered.groupby('Doc ID')['Prolog Submittal No..1'].agg(lambda x: ', '.join(x.dropna().astype(int).astype(str).str.zfill(6).unique().tolist())).reset_index()
consolidated_prolog_submittal_no.rename(columns={'Prolog Submittal No..1': 'Consolidated Prolog Submittal No'}, inplace=True)

# Merge this back into the main DataFrame
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    consolidated_prolog_submittal_no,
    on='Doc ID',
    how='left'
)

# Update the 'Prolog Submittal No.' column
df_cleaned_and_filtered['Prolog Submittal No.'] = df_cleaned_and_filtered['Consolidated Prolog Submittal No']

# Drop the temporary column
df_cleaned_and_filtered.drop(columns=['Consolidated Prolog Submittal No'], inplace=True)

print("DataFrame after updating 'Prolog Submittal No.' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Prolog Submittal No..1', 'Prolog Submittal No.']].head())

DataFrame after updating 'Prolog Submittal No.' column:


,Doc ID,Prolog Submittal No..1,Prolog Submittal No.
0,131242-WST00-PP-PM-0001,1.0,000001
1,131242-WST00-PP-PM-0001,1.0,000001
2,131242-WST00-PP-PC-0001,2.0,000002
3,131242-WSW41-PP-PC-0001,2.0,000002
4,#000002.0_ Reply_2023 08 31----,2.0,000002


## Update 'Submitted by'

### Subtask:
Update the 'Submitted by' column in `df_cleaned_and_filtered` by getting unique values from 'Document Owner' for each 'Doc ID'. If multiple different values are found, separate them with a comma.

In [42]:
# Group by 'Doc ID' and aggregate unique 'Document Owner' values
consolidated_submitted_by = df_cleaned_and_filtered.groupby('Doc ID')['Document Owner'].agg(lambda x: ', '.join(x.dropna().astype(str).unique().tolist())).reset_index()
consolidated_submitted_by.rename(columns={'Document Owner': 'Consolidated Submitted By'}, inplace=True)

# Merge this back into the main DataFrame
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    consolidated_submitted_by,
    on='Doc ID',
    how='left'
)

# Update the 'Submitted by' column
df_cleaned_and_filtered['Submitted by'] = df_cleaned_and_filtered['Consolidated Submitted By']

# Drop the temporary column
df_cleaned_and_filtered.drop(columns=['Consolidated Submitted By'], inplace=True)

print("DataFrame after updating 'Submitted by' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Document Owner', 'Submitted by']].head())

DataFrame after updating 'Submitted by' column:


,Doc ID,Document Owner,Submitted by
0,131242-WST00-PP-PM-0001,Soe Myat,"Soe Myat, Radha"
1,131242-WST00-PP-PM-0001,Radha,"Soe Myat, Radha"
2,131242-WST00-PP-PC-0001,Shena,Shena
3,131242-WSW41-PP-PC-0001,Max/Edwin,"Max/Edwin, Max"
4,#000002.0_ Reply_2023 08 31----,Max/Edwin,Max/Edwin


## Update 'Document Title'

### Subtask:
Update the 'Document Title' column in `df_cleaned_and_filtered` by getting unique values from 'Document Description / Drawing Title' for each 'Doc ID'. If multiple different values are found, separate them with a comma.

In [43]:
# Group by 'Doc ID' and aggregate unique 'Document Description / Drawing Title' values
consolidated_document_title = df_cleaned_and_filtered.groupby('Doc ID')['Document Description / Drawing Title'].agg(lambda x: ', '.join(x.dropna().astype(str).unique().tolist())).reset_index()
consolidated_document_title.rename(columns={'Document Description / Drawing Title': 'Consolidated Document Title'}, inplace=True)

# Merge this back into the main DataFrame
df_cleaned_and_filtered = pd.merge(
    df_cleaned_and_filtered,
    consolidated_document_title,
    on='Doc ID',
    how='left'
)

# Update the 'Document Title' column
df_cleaned_and_filtered['Document Title'] = df_cleaned_and_filtered['Consolidated Document Title']

# Drop the temporary column
df_cleaned_and_filtered.drop(columns=['Consolidated Document Title'], inplace=True)

print("DataFrame after updating 'Document Title' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Document Title', 'Document Description / Drawing Title']].head())

DataFrame after updating 'Document Title' column:


,Doc ID,Document Title,Document Description / Drawing Title
0,131242-WST00-PP-PM-0001,Project Quality Plan,Project Quality Plan
1,131242-WST00-PP-PM-0001,Project Quality Plan,Project Quality Plan
2,131242-WST00-PP-PC-0001,Project Risk Management Plan,Project Risk Management Plan
3,131242-WSW41-PP-PC-0001,Project Risk Management Plan,Project Risk Management Plan
4,#000002.0_ Reply_2023 08 31----,Project Risk Management Plan,Project Risk Management Plan


## Update 'This Revision'

### Subtask:
Update the 'This Revision' column in `df_cleaned_and_filtered` by assigning it the values from the 'Rev ' column.

In [44]:
# Update the 'This Revision' column with values from 'Rev '
df_cleaned_and_filtered['This Revision'] = df_cleaned_and_filtered['Rev ']

print("DataFrame after updating 'This Revision' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Rev ', 'This Revision']].head())

DataFrame after updating 'This Revision' column:


,Doc ID,Rev,This Revision
0,131242-WST00-PP-PM-0001,0,0
1,131242-WST00-PP-PM-0001,1,1
2,131242-WST00-PP-PC-0001,0,0
3,131242-WSW41-PP-PC-0001,1,1
4,#000002.0_ Reply_2023 08 31----,1,1


## Update 'This Submission Date'

### Subtask:
Update the 'This Submission Date' column in `df_cleaned_and_filtered` by assigning it the values from the 'Date Submit' column.

In [45]:
# Update the 'This Submission Date' column with values from 'Date Submit'
df_cleaned_and_filtered['This Submission Date'] = df_cleaned_and_filtered['Date Submit']

print("DataFrame after updating 'This Submission Date' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Date Submit', 'This Submission Date']].head())

DataFrame after updating 'This Submission Date' column:


,Doc ID,Date Submit,This Submission Date
0,131242-WST00-PP-PM-0001,2023-05-15,2023-05-15
1,131242-WST00-PP-PM-0001,2024-05-13,2024-05-13
2,131242-WST00-PP-PC-0001,2023-05-15,2023-05-15
3,131242-WSW41-PP-PC-0001,2023-09-05,2023-09-05
4,#000002.0_ Reply_2023 08 31----,2023-09-05,2023-09-05


## Update 'This Review Return Date'

### Subtask:
Update the 'This Review Return Date' column in `df_cleaned_and_filtered` by assigning it the values from the 'Actual Date S.O. Response' column. If 'Actual Date S.O. Response' is empty, input empty value accordingly.

In [46]:
# Update the 'This Review Return Date' column with values from 'Actual Date S.O. Response'
df_cleaned_and_filtered['This Review Return Date'] = df_cleaned_and_filtered['Actual Date S.O. Response']

print("DataFrame after updating 'This Review Return Date' column:")
display(df_cleaned_and_filtered[['Doc ID', 'Actual Date S.O. Response', 'This Review Return Date']].head())

DataFrame after updating 'This Review Return Date' column:


,Doc ID,Actual Date S.O. Response,This Review Return Date
0,131242-WST00-PP-PM-0001,2023-12-05,2023-12-05
1,131242-WST00-PP-PM-0001,2024-09-10,2024-09-10
2,131242-WST00-PP-PC-0001,2023-06-14,2023-06-14
3,131242-WSW41-PP-PC-0001,2023-09-13,2023-09-13
4,#000002.0_ Reply_2023 08 31----,2023-09-13,2023-09-13


## Update 'This Revision Approval Status'

### Subtask:
Update the 'This Revision Approval Status' column in `df_cleaned_and_filtered` based on the 'SO Review Status' using the following rules:
- "REJ" for "Rejected" or "Not Approved - Revise and resubmit"
- "AWC" for "Approved with Comments" or "Approved as noted"
- "APP" for "For Information" or "Approved"
- "Pending" for "Awaiting S.O.'s response", or if 'Latest Approval Status' is empty or 0
- "VOID" for "(VOID / NOT IN USE)"
- "To Check" for any other cases.

In [47]:
# The 'get_approval_code' function is already defined and can be reused
df_cleaned_and_filtered['This Revision Approval Status'] = df_cleaned_and_filtered['SO Review Status'].apply(get_approval_code)

print("DataFrame after updating 'This Revision Approval Status' column:")
display(df_cleaned_and_filtered[['Doc ID', 'SO Review Status', 'This Revision Approval Status']].head())

DataFrame after updating 'This Revision Approval Status' column:


,Doc ID,SO Review Status,This Revision Approval Status
0,131242-WST00-PP-PM-0001,Approved with Comments,AWC
1,131242-WST00-PP-PM-0001,Approved with Comments,AWC
2,131242-WST00-PP-PC-0001,Approved with Comments,AWC
3,131242-WSW41-PP-PC-0001,Approved with Comments,AWC
4,#000002.0_ Reply_2023 08 31----,Approved with Comments,AWC


## Format Date Columns to 'dd/mm/yyyy'

### Subtask:
Iterate through all columns in `df_cleaned_and_filtered` and format any datetime columns to 'dd/mm/yyyy'.

In [50]:
for col in df_cleaned_and_filtered.columns:
    if pd.api.types.is_datetime64_any_dtype(df_cleaned_and_filtered[col]):
        # Apply the desired date format and convert to string
        df_cleaned_and_filtered[col] = df_cleaned_and_filtered[col].dt.strftime('%d/%m/%Y')

print("DataFrame after formatting date columns (first 5 rows):")
display(df_cleaned_and_filtered.head())

DataFrame after formatting date columns (first 5 rows):


,Doc ID,1st Submission\nDate,Latest Submission Date,Latest Revision,Latest Approval Status,Approval Code,# of Submissions,Prolog Submittal No.,Submitted by,Document Title,...,SO Response Variance,SO Review Status,To Resubmit (Yes/No),Date CES to Response\n(14 Working Days),Closed,Overdue to resubmit,Delays to resubmit,Target date to resubmit,Remark,Recalculated Doc ID
0,131242-WST00-PP-PM-0001,15/05/2023,13/05/2024,Rev.1,Approved with Comments,AWC,2,000001,"Soe Myat, Radha",Project Quality Plan,...,122,Approved with Comments,YES,26/12/2023,YES,Resubmitted,NaN,NaN,NaN,131242-WST00-PP-PM-0001
1,131242-WST00-PP-PM-0001,15/05/2023,13/05/2024,Rev.1,Approved with Comments,AWC,2,000001,"Soe Myat, Radha",Project Quality Plan,...,69,Approved with Comments,YES,30/09/2024,NO,Overdue,343.0,NaN,NaN,131242-WST00-PP-PM-0001
2,131242-WST00-PP-PC-0001,15/05/2023,15/05/2023,Rev.0,Approved with Comments,AWC,1,000002,Shena,Project Risk Management Plan,...,1,Approved with Comments,YES,05/07/2023,YES,Resubmitted,343.0,NaN,NaN,131242-WST00-PP-PC-0001
3,131242-WSW41-PP-PC-0001,05/09/2023,30/10/2023,Rev.2,Approved,APP,2,000002,"Max/Edwin, Max",Project Risk Management Plan,...,1,Approved with Comments,YES,03/10/2023,YES,Resubmitted,343.0,NaN,NaN,131242-WSW41-PP-PC-0001
4,#000002.0_ Reply_2023 08 31----,05/09/2023,05/09/2023,Rev.1,Approved with Comments,AWC,1,000002,Max/Edwin,Project Risk Management Plan,...,1,Approved with Comments,YES,03/10/2023,YES,Resubmitted,343.0,NaN,NaN,#000002.0_ Reply_2023 08 31-WSW41-PP-PC-0001


## Export DataFrame to Excel

### Subtask:
Export the `df_cleaned_and_filtered` DataFrame to an Excel file and provide a file selection window for download.

In [51]:
from google.colab import files

output_file_name = 'Processed_Submittal_Tracker.xlsx'

# Save the DataFrame to an Excel file
df_cleaned_and_filtered.to_excel(output_file_name, index=False)

print(f"DataFrame successfully saved to '{output_file_name}'.")
print("A download prompt will appear shortly.")

# Trigger the download
files.download(output_file_name)

DataFrame successfully saved to 'Processed_Submittal_Tracker.xlsx'.
A download prompt will appear shortly.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>